[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/PDF_Creation/PDF_Creator.ipynb)

Open this exact notebook in Colab, connect to a runtime, and run cells from the top using ▶. GitHub Preview displays the notebook without executing it.


## PDF creation

Compile the existing LaTeX sources without generating or rewriting their manuscript content. Each paper has its own execution cell. Existing PDFs remain recoverable when compilation fails.


## Failure reporting
Each operation reports what failed, the exception reason, and what to assess. Unexpected failures retain their traceback and stop execution. A missing project dataset is handled separately with a debug message and skipped dependent calculations.

In [ ]:
from contextlib import contextmanager
import subprocess

@contextmanager
def report_failure(stage, assessment):
    try:
        yield
    except Exception as error:
        print(f"FAILED: {stage}", flush=True)
        print(f"Reason: {type(error).__name__}: {error}", flush=True)
        if isinstance(error, subprocess.CalledProcessError):
            details = error.stderr or error.stdout or error.output
            if details:
                if isinstance(details, bytes):
                    details = details.decode("utf-8", errors="replace")
                print(f"Process output (last 4000 characters):\n{str(details)[-4000:]}", flush=True)
        print(f"Assess: {assessment}", flush=True)
        print("This operation failed. Correct the cause and rerun it before dependent cells.", flush=True)
        raise


## Optional dependency installation

In a fresh Colab runtime, set `INSTALL_DEPENDENCIES = True` below and run the installation cells. In an already prepared workspace, leave it `False`: no packages will be installed. Reopening the same Codespace retains installed packages; a new or rebuilt environment needs setup again. Restart the kernel if an installation replaces a package already imported.


In [ ]:
INSTALL_DEPENDENCIES = False


## LaTeX and PDF libraries

Prepare the typesetting engine and PDF parser used to compile and validate documents.


In [ ]:
if globals().get("INSTALL_DEPENDENCIES", True):
    with report_failure('LaTeX and PDF libraries', 'Check network access, package availability, Python compatibility, installation permissions and available disk space.'):
        import importlib.metadata
        import subprocess
        import sys
        import shutil
        import os
        try:
            installed = importlib.metadata.version("PyMuPDF")
        except importlib.metadata.PackageNotFoundError:
            installed = None
        if installed != "1.26.6":
            subprocess.run([sys.executable,"-m","pip","install","PyMuPDF==1.26.6"],check=True)
        if not shutil.which("xelatex"):
            if shutil.which("apt-get") and hasattr(os,"geteuid") and os.geteuid()==0:
                subprocess.run(["apt-get","update","-qq"],check=True)
                subprocess.run(["apt-get","install","-y","texlive-xetex","texlive-latex-extra","texlive-pictures","texlive-fonts-recommended","lmodern"],check=True)
            else:
                raise RuntimeError("Install XeLaTeX, LaTeX extra packages, PGFPlots and Latin Modern fonts before compiling.")
else:
    print('Installation skipped. Using the existing environment.')


## Safe PDF replacement

Compile in a temporary directory, validate the generated PDF, then replace the destination atomically. A previous PDF is backed up and the backup is removed only after success.


In [ ]:
with report_failure('Safe PDF replacement', 'Check the TeX process output, source and figure paths, fonts, installed XeLaTeX packages and output permissions. Preserve the existing PDF and backup until compilation succeeds.'):
    from pathlib import Path
    import os
    import shutil
    import subprocess
    import tempfile
    import fitz

    ROOT = Path(os.environ.get("ADVANCED_DATA_SCIENCE_ROOT", Path.cwd())).expanduser().resolve()
    for candidate in (ROOT, *ROOT.parents):
        if (candidate / "PDF_Creation" / "Projects_LaTeX").is_dir():
            ROOT = candidate
            break
    LATEX_DIRECTORY = ROOT / "PDF_Creation" / "Projects_LaTeX"
    PDF_DIRECTORY = ROOT / "PDFs"
    FIGURE_DIRECTORY = ROOT / "figure_outputs"

    def create_pdf(source, destination, figures=None):
        source, destination = Path(source).resolve(), Path(destination).resolve()
        if not source.is_file():
            raise FileNotFoundError(source)
        if not shutil.which("xelatex"):
            raise RuntimeError("Install XeLaTeX using the dependencies cell.")
        destination.parent.mkdir(parents=True, exist_ok=True)
        backup = destination.with_name(destination.stem + ".backup.pdf")
        # A prior failed attempt may already have a recoverable backup.
        if backup.exists() and not destination.exists():
            shutil.copy2(backup, destination)
        if destination.exists():
            shutil.copy2(destination, backup)
        try:
            with tempfile.TemporaryDirectory(prefix="pdf-build-", dir=destination.parent) as temporary:
                build = Path(temporary)
                shutil.copytree(source.parent, build, dirs_exist_ok=True)
                if figures is not None and Path(figures).is_dir():
                    shutil.copytree(figures, build / "figures", dirs_exist_ok=True)
                for _ in range(2):
                    result = subprocess.run(["xelatex", "-interaction=nonstopmode", "-halt-on-error", source.name],
                                            cwd=build, capture_output=True, text=True)
                    if result.returncode:
                        raise RuntimeError("LaTeX compilation failed; the existing PDF and backup are preserved.\n" + result.stdout[-6000:])
                generated = build / source.with_suffix(".pdf").name
                with fitz.open(generated) as document:
                    if document.page_count < 1 or document.is_encrypted:
                        raise RuntimeError("The generated PDF is not a readable document.")
                generated.replace(destination)
            if backup.exists():
                backup.unlink()
        except BaseException:
            if backup.exists() and not destination.exists():
                shutil.copy2(backup, destination)
            raise
        print(f"PDF saved: {destination}")
        return destination


## From Linguistic Context to Semantic Structure

Compile this paper’s LaTeX source with the explicitly configured figure directory. Missing figures or compilation errors preserve the previous PDF. The current manuscript remains a draft until its scientific results are finalized.


In [ ]:
with report_failure('From Linguistic Context to Semantic Structure', 'Check the TeX process output, source and figure paths, fonts, installed XeLaTeX packages and output permissions. Preserve the existing PDF and backup until compilation succeeds.'):
    create_pdf(
        LATEX_DIRECTORY / "From_Linguistic_Context_to_Semantic_Structure.tex",
        PDF_DIRECTORY / "From_Linguistic_Context_to_Semantic_Structure.pdf",
        figures=FIGURE_DIRECTORY,
    )
